# **Customer_360_Intelligence_Copilot**

## End-to-end customer analytics and AI support project

### **Project scenario:** 
a synthetic grocery quick-commerce business. This notebook documents an end-to-end workflow for ingesting, inspecting, cleaning, validating, storing, and analysing operational data, then using the trusted data in customer analytics and AI-assisted support workflows.
### Project setup
1. Run `python generate_data.py` from the repository root.
2. Install the required packages with `pip install -r requirements.txt`.
3. Run this notebook from the repository root so that the relative paths resolve correctly.

**All data and policies in this project are fictional. LLM interactions are handled locally through `mock_llm.py`, so no API key is required.**
### Project objectives
- Build a repeatable data-ingestion and cleaning workflow.
- Store trusted datasets in CSV files and SQLite.
- Explore customer value and predict customer lapse.
- Triage support tickets while applying privacy safeguards.
- Retrieve relevant policy information and demonstrate controlled refund-tool workflows.
The notebook combines data engineering, analytics, machine learning, NLP, retrieval-augmented generation (RAG), and tool-based automation in one reproducible project.

## **0. Project structure and end-to-end workflow**
generate_data.py → data/raw/*.csv → data audit → cleaning and validation → data/clean/*.csv + data/instamart.db → RFM analysis and customer segments → lapse prediction → privacy-aware ticket triage → policy retrieval → refund tools.

 **Data-ingestion approach:** data ingestion does not always require a live API call. In this project, the source is a set of local CSV exports. A production implementation could use an approved API or data-warehouse connector, with credentials and network access managed securely.

In [1]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if not (ROOT / "generate_data.py").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
print("Project root:", ROOT)
print("Root files:", sorted(p.name for p in ROOT.iterdir() if p.is_file()))

Project root: c:\Users\User\Desktop\Swiggy Instamart\Swiggy Instamart
Root files: ['.gitignore', 'Customer_360_Intelligence_Copilot.ipynb', 'README.md', 'Swiggy Instamart.code-workspace', 'generate_data.py', 'mock_llm.py', 'requirements.txt', 'run_part_d.py', 'run_project.py']


#### Things to Note — project setup
- **Why establish a project root?** It gives the notebook a consistent base path for locating scripts, data folders, and source modules.
- **Why add the root to sys.path?** It allows Python to import project modules from the src directory when the notebook is run from the repository.
- **Why run from the repository root?** The project uses relative paths; running elsewhere can cause file-not-found errors.
- **What makes the data reproducible?** The generator uses a fixed random seed, so the same synthetic inputs can be recreated.

### **1. Generate the raw source data**
The data generator uses a fixed random seed, making the synthetic dataset reproducible. It creates customer, product, order, support-ticket, evaluation-question, and agent-scenario inputs, along with policy text files.

Run the generator when setting up the project from a clean copy. Running it again recreates the same synthetic source exports.

In [4]:
import subprocess, sys
subprocess.run([sys.executable, str(ROOT / "generate_data.py")], check=True)
raw_dir = ROOT / "data" / "raw"
print("Raw files:", sorted(p.name for p in raw_dir.iterdir()))

Raw files: ['agent_scenarios.csv', 'customers.csv', 'orders.csv', 'policy_eval.csv', 'products.csv', 'tickets.csv']


### **Things to Note — generating data**
- **What is data generation in this project?** It creates synthetic CSV exports and policy/scenario files that act as the source system.
- **Is this a live API fetch?** No. The current source is local generated data. A production ingestion layer could later connect to an approved API or warehouse.
- **Why use synthetic data?** It allows the workflow to be demonstrated without exposing real customer information.
- **Why is a fixed random seed useful?** It makes runs repeatable and makes debugging and result comparison easier.

### **2. Load the source exports and inspect their structure**
The CSV files are loaded into pandas DataFrames so that their dimensions, sample records, data types, missing values, and blank values can be inspected before cleaning.

keep_default_na=False is used when reading ticket text. This preserves empty strings as empty strings, allowing the ticket-triage workflow to identify empty messages explicitly rather than silently converting their representation during ingestion.

In [5]:
import pandas as pd
customers_raw = pd.read_csv(raw_dir / "customers.csv", keep_default_na=False)
products_raw = pd.read_csv(raw_dir / "products.csv", keep_default_na=False)
orders_raw = pd.read_csv(raw_dir / "orders.csv", keep_default_na=False)
tickets_raw = pd.read_csv(raw_dir / "tickets.csv", keep_default_na=False)

for name, df in [("customers", customers_raw), ("products", products_raw),
                 ("orders", orders_raw), ("tickets", tickets_raw)]:
    print(f"\\n{name}: shape={df.shape}")
    display(df.head(3))
    display(pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "missing_count": df.isna().sum(),
        "blank_count": df.astype(str).apply(lambda col: col.str.strip().eq("").sum()),
        "missing_pct": (df.isna().mean() * 100).round(2)
    }))

\ncustomers: shape=(1200, 5)


,customer_id,city,signup_date,age,gender
0,C0001,Chennai,2024-01-13,35,MALE
1,C0002,DELHI,2024-02-17,31,MALE
2,C0003,bengaluru,2025-02-18,18,F


,dtype,missing_count,blank_count,missing_pct
customer_id,str,0,0,0.0
city,str,0,0,0.0
signup_date,str,0,0,0.0
age,str,0,61,0.0
gender,str,0,30,0.0


\nproducts: shape=(40, 4)


,product_id,product_name,category,price_inr
0,P001,Basmati Rice 1kg,Grocery,129
1,P002,Toor Dal 1kg,Grocery,165
2,P003,Atta 5kg,Grocery,245


,dtype,missing_count,blank_count,missing_pct
product_id,str,0,0,0.0
product_name,str,0,0,0.0
category,str,0,0,0.0
price_inr,int64,0,0,0.0


... (2 outputs left out)

,dtype,missing_count,blank_count,missing_pct
order_id,str,0,0,0.0
customer_id,str,0,0,0.0
product_id,str,0,0,0.0
quantity,int64,0,0,0.0
order_ts,str,0,0,0.0
status,str,0,0,0.0
discount_pct,int64,0,0,0.0
order_amount,str,0,0,0.0
delivery_minutes,str,0,2541,0.0


\ntickets: shape=(164, 5)


,ticket_id,customer_id,order_id,text,true_category
0,T0024,C0486,ORD-105035,There is an unauthorized transaction on my car...,fraud
1,T0073,C0080,ORD-100816,The invoice for ORD-100816 shows a wrong amoun...,billing
2,T0095,C1087,ORD-111156,The payment page shows an error and the app fr...,technical


,dtype,missing_count,blank_count,missing_pct
ticket_id,str,0,0,0.0
customer_id,str,0,0,0.0
order_id,str,0,8,0.0
text,str,0,4,0.0
true_category,str,0,0,0.0


### **Data-quality checks to investigate**
During the initial audit, I check:
- Which fields are expected to act as unique identifiers?
- Whether repeated records are exact duplicates or later states of the same order.
- Whether values are invalid, such as negative delivery duration, non-numeric monetary values, or implausible ages.
- Whether a field contains a true missing value (NaN) or an empty string.
- What risks would arise if the raw data were used directly for modelling.

In [9]:
from src.data_pipeline import audit, clean_data, validate
audit()


customers: shape=(1200, 5)
customer_id    str
city           str
signup_date    str
age            str
gender         str
dtype: object
             missing_count  missing_pct
customer_id              0          0.0
city                     0          0.0
signup_date              0          0.0
age                      0          0.0
gender                   0          0.0

products: shape=(40, 4)
product_id        str
product_name      str
category          str
price_inr       int64
dtype: object
              missing_count  missing_pct
product_id                0          0.0
product_name              0          0.0
category                  0          0.0
price_inr                 0          0.0

orders: shape=(13084, 9)
order_id              str
customer_id           str
product_id            str
quantity            int64
order_ts              str
status                str
discount_pct        int64
order_amount          str
delivery_minutes      str
dtype: object
                 

### **Things to Note — data inspection and audit**
- **What is a unique key?** A field, or combination of fields, intended to identify one entity or record uniquely (for example, a customer ID or order ID).
- **Are all duplicate rows the same?** No. Exact duplicate rows are repeated copies; multiple rows for one order may represent different states or updates. The pipeline removes exact duplicates and then keeps the latest order record.
- **What are examples of invalid values?** Negative delivery duration, non-numeric order amounts, and ages outside a plausible range.
- **What is the difference between blank and missing?** A blank is an empty text value (""); a missing value is represented as null/`NaN`. They may require different handling.
- **Why audit before modelling?** Bad types, duplicates, impossible values, and missing data can distort features, labels, model training, and evaluation.

### **3. Clean the data and create trusted outputs**
**The cleaning pipeline applies explicit transformation rules:**
- Standardise city and gender values.
- Treat impossible ages as missing, then fill them using the median for the relevant city.
- Parse timestamps, remove exact duplicate rows, and retain the latest record for each order ID.
- Convert invalid order amounts to missing values and calculate replacements using product price × quantity × discount.
- Replace negative delivery durations with missing values and impute delivery times only for completed orders, using product-category medians.
- Save the cleaned datasets as CSV files and persist the resulting tables in SQLite.

A cancelled order should retain a missing delivery duration because no completed delivery duration exists to measure. Filling it with a typical delivery time would create a misleading value.

In [10]:
clean = clean_data()
for name, df in clean.items():
    print(name, df.shape)
    display(df.head(3))
validate()

customers (1200, 5)


,customer_id,city,signup_date,age,gender
0,C0001,Chennai,2024-01-13,35,male
1,C0002,Delhi,2024-02-17,31,male
2,C0003,Bengaluru,2025-02-18,18,female


products (40, 4)


,product_id,product_name,category,price_inr
0,P001,Basmati Rice 1kg,Grocery,129
1,P002,Toor Dal 1kg,Grocery,165
2,P003,Atta 5kg,Grocery,245


orders (12309, 9)


,order_id,customer_id,product_id,quantity,order_ts,status,discount_pct,order_amount,delivery_minutes
827,ORD-100001,C0001,P001,1,2025-09-16 08:07:21,completed,15,109.65,20
4294,ORD-100002,C0001,P008,1,2025-09-03 07:34:01,completed,40,25.20,23
9973,ORD-100003,C0001,P008,1,2025-04-08 17:08:14,completed,0,42.00,23


ALL CHECKPOINTS PASSED


### **Things to Note — cleaning and validation**
- **Why standardise city and gender?** Values such as different capitalisations or spellings should not become separate categories accidentally.
- Why use the city median for age? The median is less sensitive to extreme values than the mean and uses a relevant city-level group for imputation.
- **Why keep the latest order row?** It represents the most recent recorded state when an order ID has multiple updates.
- **Why calculate a replacement order amount?** Product price, quantity, and discount provide a rule-based way to reconstruct an invalid amount when the required inputs are available.
- **Why impute delivery time only for completed orders?** A cancelled order has no actual completed delivery duration; filling one would imply a delivery that did not happen.
- **Why validate after cleaning?** Validation checks whether the transformed datasets satisfy expected quality rules before they are used downstream.

### **4. Query the cleaned SQLite database**
The cleaned tables are stored in `data/instamart.db`. SQLite provides a lightweight relational database that can be queried without running a separate database server.

The query below joins orders with customer and product information, calculates revenue for completed orders, and groups the result by city and product category. I compare the SQL result with pandas totals as a basic consistency check.

In [11]:
import sqlite3
import pandas as pd
from pathlib import Path

ROOT = Path(".")
db_path = ROOT / "data" / "instamart.db"

with sqlite3.connect(db_path) as con:
    tables = pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name", con)
    display(tables)
    
    q2 = pd.read_sql_query("""
        SELECT c.city, p.category, ROUND(SUM(o.order_amount), 2) AS revenue
        FROM orders o
        JOIN customers c USING(customer_id)
        JOIN products p USING(product_id)
        WHERE o.status = 'completed'
        GROUP BY c.city, p.category
        ORDER BY c.city, p.category
    """, con)

display(q2)
print("SQL grand total INR:", round(q2.revenue.sum(), 2))

,name
0,customers
1,orders
2,products


,city,category,revenue
0,Bengaluru,Dairy,87075.05
1,Bengaluru,Electronics,233909.45
2,Bengaluru,Grocery,110915.65
3,Bengaluru,Personal Care,20193.85
4,Bengaluru,Snacks,71549.40
5,Chennai,Dairy,78301.80
6,Chennai,Electronics,195508.80
7,Chennai,Grocery,93817.45
8,Chennai,Personal Care,27675.40
9,Chennai,Snacks,68325.70


SQL grand total INR: 2998668.8


### **Things to Note — SQL and SQLite**
- **What is SQLite?** A lightweight relational database stored in a local file; it does not require a separate database server.
- **Why use joins?** The order table contains transaction facts, while customer and product tables provide descriptive attributes such as city and category.
-** What does SUM(o.order_amount) calculate?** It adds order amounts within each city/category group.
- **Why filter status = 'completed'?** The query is calculating completed-order revenue, not revenue from cancelled or other order states.
- **Why compare SQL with pandas?** Independent calculations provide a useful consistency check and can reveal join, filter, or aggregation mistakes.

## **5. Customer segmentation and lapse prediction**
RFM analysis summarises customer purchasing behaviour using three measures:
- **Recency:** number of days since the customer's most recent completed order at the analysis snapshot date.
- **Frequency:** number of distinct completed orders.
- **Monetary:** total spend across completed orders.

The modelling module scales the RFM features for K-means clustering, compares candidate values of K, uses PCA to inspect explained variance, and builds a customer-lapse model.

For lapse prediction, features are calculated using orders before 2025-09-01; September is reserved for defining the target label. The scaler and encoder are fitted within the training pipeline so that information from the test data does not leak into preprocessing.

In [12]:
from src.customer_models import rfm_and_segments, build_lapse_model
rfm, k_diagnostics, segment_profile, pca_variance = rfm_and_segments(k=4)
display(k_diagnostics.round(3))
display(segment_profile.round(2))
print("PCA explained variance ratio:", pca_variance, "total:", pca_variance.sum())
display(rfm.head())
lapse_summary = build_lapse_model()
lapse_summary

,k,inertia,silhouette
0,2,1820.761,0.465
1,3,1268.235,0.445
2,4,743.071,0.526
3,5,617.367,0.480
4,6,503.171,0.482
5,7,408.460,0.416
6,8,366.503,0.412


,segment_id,size,recency,frequency,monetary
0,0,436,28.86,3.77,1057.62
1,1,138,24.12,6.77,7129.70
2,2,419,8.77,19.40,3410.02
3,3,182,109.04,2.20,685.99


PCA explained variance ratio: [0.63427875 0.21803762] total: 0.8523163667502166


,customer_id,frequency,monetary,recency,segment_id,segment_name,is_premium
0,C0001,4,512.85,14,0,At Risk,False
1,C0002,7,4644.90,28,1,Premium,True
2,C0003,22,3644.85,7,2,Frequent,False
3,C0004,5,369.00,0,0,At Risk,False
4,C0005,2,324.00,147,3,At Risk,False


{'cohort_rows': 1156,
 'lapsed': 443,
 'baseline_accuracy': 0.6163793103448276,
 'models': {'logistic': {'best_params': {'model__C': 100},
   'cv_f1': np.float64(0.650552962900789),
   'precision': 0.6444444444444445,
   'recall': 0.651685393258427,
   'f1': 0.6480446927374302,
   'confusion_matrix': [[111, 32], [31, 58]]},
  'forest': {'best_params': {'model__max_depth': 8,
    'model__min_samples_leaf': 10},
   'cv_f1': np.float64(0.6865517479789224),
   'precision': 0.6333333333333333,
   'recall': 0.6404494382022472,
   'f1': 0.6368715083798883,
   'confusion_matrix': [[110, 33], [32, 57]]}},
 'retention_list':     customer_id  lapse_probability  actually_lapsed
 13        C0862           0.510513                0
 190       C0267           0.477212                0
 165       C1008           0.473260                0
 227       C0060           0.471746                0
 5         C0802           0.471582                0
 216       C0942           0.465245                0
 144   

### **Things to Note — RFM, clustering, and lapse prediction**
- **What does RFM stand for?** Recency, Frequency, and Monetary value; together they describe how recently, how often, and how much a customer purchases.
- **Why scale RFM before K-means?** K-means uses distances. Scaling prevents a feature with a larger numeric range from dominating the distance calculation.
- **What is K-means?** An unsupervised clustering algorithm that groups observations around K cluster centres based on feature similarity.
- **Why compare different K values?** The number of clusters is a modelling choice; diagnostics help assess candidate cluster counts rather than assuming one value is automatically best.
- **What is PCA?** Principal Component Analysis transforms correlated features into fewer components and reports how much variance those components explain.
-** What is lapse prediction?** A supervised classification task that estimates whether a customer will meet the project's lapse definition.
- What is data leakage? When information unavailable at prediction time, or information from the test/target period, enters model features or preprocessing and makes evaluation unrealistically optimistic.
- **Why exclude September orders from features?** September is used to define the target label. Including those orders in the features would reveal information from the period being predicted.
- **Why is the Premium-only list illustrative?** Its Premium flag uses whole-window RFM, which includes information beyond the pre-September feature window; it is not a strictly time-safe prospective retention list.
- **Why can accuracy mislead?** A model can score well by favouring the majority class while performing poorly on the customers who actually lapse. Precision, recall, F1, and a confusion matrix provide additional insight.

### **Understanding data leakage and model evaluation**
The modelling design raises four important questions:
1. Which dates are allowed in the lapse-model features?
2. Why are September orders excluded from the input features?
3. Why should a Premium-only retention list be described as illustrative when the Premium flag is calculated using the full observation window?
4. Why can accuracy give an incomplete picture when a substantial minority of customers lapse?

## **6. Support-ticket triage**
The ticket-triage pipeline demonstrates several safeguards and processing steps:
1. Input guardrails block empty and overlong messages.
2. Personally identifiable information (PII) is redacted in the order email → phone → order ID.
3. Retry logic handles simulated transient errors.
4. Predicted categories are validated against an allowed set, with a keyword-based fallback.
5. Token budgets are monitored and selected cases are routed for human review.

Only the redacted ticket text is sent as the user message to the local mock classifier.

In [13]:
from src.triage import run_pipeline, ZERO

# Run zero-shot classification with a 6,000-token budget
budget_results, budget_summary = run_pipeline(
    prompt=ZERO,
    token_budget=6000
)

# Display summary
print("Budget summary:")
print(budget_summary)

# Count exhausted tickets
exhausted = (
    budget_results["status"] == "budget_exhausted"
).sum()

print("\nTickets stopped due to budget:", exhausted)

# Verify assignment requirement
assert exhausted > 0, "No tickets exhausted the budget!"

print("\nCSV saved: outputs/triage_budget_demo.csv")

retry ticket=T0002 retry=0 wait=0.0530s
retry ticket=T0116 retry=0 wait=0.0453s
retry ticket=T0153 retry=0 wait=0.0484s
retry ticket=T0039 retry=0 wait=0.0478s
retry ticket=T0152 retry=0 wait=0.0542s
retry ticket=T0043 retry=0 wait=0.0535s
retry ticket=T0111 retry=0 wait=0.0562s
retry ticket=T0155 retry=0 wait=0.0461s
retry ticket=T0100 retry=0 wait=0.0503s
retry ticket=T0068 retry=0 wait=0.0454s
retry ticket=T0135 retry=0 wait=0.0477s
retry ticket=T0009 retry=0 wait=0.0513s
retry ticket=T0067 retry=0 wait=0.0453s
retry ticket=T0106 retry=0 wait=0.0475s
retry ticket=T0097 retry=0 wait=0.0531s
Budget summary:
{'tokens': 6009, 'redactions': {'email': 55, 'phone': 91, 'order_id': 132}, 'status_counts': {'classified_json': 111, 'budget_exhausted': 29, 'classified_keyword': 9, 'flagged_invalid_output': 7, 'skipped_empty': 4, 'flagged_long': 4}}

Tickets stopped due to budget: 29

CSV saved: outputs/triage_budget_demo.csv


In [14]:
from src.triage import run_pipeline, FEW, ZERO
triage_results, triage_summary = run_pipeline(prompt=FEW, token_budget=100000)
print(triage_summary)
display(triage_results.head(10))
display(triage_results.status.value_counts().rename_axis("status").reset_index(name="tickets"))

retry ticket=T0002 retry=0 wait=0.0530s
retry ticket=T0116 retry=0 wait=0.0453s
retry ticket=T0153 retry=0 wait=0.0484s
retry ticket=T0039 retry=0 wait=0.0478s
retry ticket=T0152 retry=0 wait=0.0542s
retry ticket=T0043 retry=0 wait=0.0535s
retry ticket=T0111 retry=0 wait=0.0562s
retry ticket=T0155 retry=0 wait=0.0461s
retry ticket=T0100 retry=0 wait=0.0503s
retry ticket=T0068 retry=0 wait=0.0454s
retry ticket=T0135 retry=0 wait=0.0477s
retry ticket=T0009 retry=0 wait=0.0513s
retry ticket=T0067 retry=0 wait=0.0453s
retry ticket=T0106 retry=0 wait=0.0475s
retry ticket=T0097 retry=0 wait=0.0531s
retry ticket=T0003 retry=0 wait=0.0518s
retry ticket=T0019 retry=0 wait=0.0478s
retry ticket=T0086 retry=0 wait=0.0524s
retry ticket=T0065 retry=0 wait=0.0551s
{'tokens': 12520, 'redactions': {'email': 55, 'phone': 91, 'order_id': 132}, 'status_counts': {'classified_json': 135, 'classified_keyword': 12, 'flagged_invalid_output': 9, 'skipped_empty': 4, 'flagged_long': 4}}


,ticket_id,customer_id,status,category,segment_name,priority,route,tokens_used
0,T0024,C0486,classified_json,fraud,At Risk,HIGH,security_team,84
1,T0073,C0080,classified_json,billing,Frequent,NORMAL,support_queue,80
2,T0095,C1087,classified_json,technical,At Risk,NORMAL,support_queue,82
3,T0159,C1172,skipped_empty,,Frequent,NORMAL,human_review,0
4,T0080,C0910,classified_json,fraud,At Risk,HIGH,security_team,84
5,T0124,C0139,classified_json,fraud,Frequent,HIGH,security_team,84
6,T0061,C0812,classified_json,billing,Frequent,NORMAL,support_queue,76
7,T0118,C0355,classified_json,delivery,Frequent,NORMAL,support_queue,79
8,T0037,C1156,classified_json,billing,Premium,HIGH,premium_desk,80
9,T0002,C0881,classified_json,delivery,At Risk,NORMAL,support_queue,78


,status,tickets
0,classified_json,135
1,classified_keyword,12
2,flagged_invalid_output,9
3,skipped_empty,4
4,flagged_long,4


### **Things to Note — support-ticket triage**
- **Why block empty tickets?** There is no meaningful message to classify, so a model call would add no value.
- **Why block overlong tickets?** This enforces input limits and helps control token usage and processing cost.
- **What is PII?** Personally identifiable information, such as email addresses, phone numbers, or order identifiers in this project's redaction examples.
- **Why redact in a defined order?** Ordered redaction helps prevent one pattern from interfering with another and makes the processing sequence predictable.
- **Why retry transient errors?** A temporary failure may succeed on a later attempt; retries improve resilience for recoverable errors.
- **Why validate categories?** A model response outside the allowed category set should not be trusted as a valid classification.
- **Why use a keyword fallback?** It provides a simple backup route when the simulated classifier fails or returns an invalid category; uncertain cases can still be sent for human review.
- **Why keep a human-review route?** Ambiguous, invalid, or sensitive cases should not be resolved through unsupported model guesses.

### **Privacy, validation, and safe routing**
The triage workflow is designed around conservative handling:
- PII redaction is a defence-in-depth measure, not a guarantee that every sensitive detail will be removed.
- Invalid or uncertain categories should be routed to human review rather than guessed.
- Fraud indicators should take priority over ordinary customer-segment routing.
- Empty or excessively long tickets should be stopped before a model call to avoid meaningless processing, unnecessary cost, and avoidable exposure of input data.

## **7. Policy retrieval with RAG**
The retrieval component splits policy documents into chunks, fits a TF-IDF vectorizer, and ranks the chunks using cosine similarity. The retrieved policy text is then passed to the local policy assistant.

The assistant is expected to answer using the retrieved context only. If the available policy context does not support an answer, it should use the configured fallback rather than inventing a policy.

In [15]:
from src.policy_rag import PolicyRAG, evaluate
rag = PolicyRAG(chunk_size=40, overlap=10)
print("Policy chunks:", len(rag.chunks))
display(pd.DataFrame(rag.chunks).head())
print(rag.answer("What delivery time makes a completed order eligible for a full refund?"))
policy_eval = evaluate(rag)
display(policy_eval)
print("Expected-document hits:", int(policy_eval.retrieved_doc.sum()))
print("Expected-fact hits:", int(policy_eval.fact_found.sum()))
print("Fallbacks:", int(policy_eval.fallback.sum()))

Policy chunks: 15


,source,text
0,delivery_policy.txt,Delivery Policy. Standard delivery is promised...
1,delivery_policy.txt,"of INR 25 is charged on orders below INR 199, ..."
2,delivery_policy.txt,60 minutes and the app shows a rain notice. A ...
3,escalation_policy.txt,Escalation Policy. A ticket about fraud or an ...
4,escalation_policy.txt,"1 hour, and all other tickets within 24 hours...."


{'answer': 'order is delivered in more than 45 minutes, the customer receives a full refund of the order amount.', 'sources': ['refund_policy.txt', 'privacy_policy.txt'], 'retrieved': [{'source': 'refund_policy.txt', 'score': 0.35464416504570484, 'text': 'order is delivered in more than 45 minutes, the customer receives a full refund of the order amount. If a completed order is delivered in 45 minutes or less, no late-delivery refund is paid. Refunds up to INR 1000 are'}, {'source': 'refund_policy.txt', 'score': 0.31425507733397284, 'text': 'Refund Policy. A refund can be requested only for orders whose status is completed. Orders with status cancelled were never charged, so no refund applies to them. If a completed order is delivered in more than 45 minutes, the customer'}, {'source': 'privacy_policy.txt', 'score': 0.15759666156595936, 'text': 'Delivery addresses are deleted from active systems 180 days after the last order. A customer can ask for account deletion and the request is c

,question,expected_doc,retrieved_doc,fact_found,fallback
0,What delivery time makes a completed order eli...,refund_policy.txt,True,True,False
1,Who must approve a refund above INR 1000?,refund_policy.txt,True,True,False
2,How many days do I have to raise a refund requ...,refund_policy.txt,True,True,False
3,What is the delivery fee on small orders?,delivery_policy.txt,True,True,False
4,On orders of what value is delivery free?,delivery_policy.txt,True,True,False
5,Why does the electronics delivery need a one t...,delivery_policy.txt,True,True,False
6,Is cash on delivery available for a large order?,payment_policy.txt,True,True,False
7,How soon is money returned when a payment fail...,payment_policy.txt,True,True,False
8,Can wallet balance be refunded as cash?,payment_policy.txt,True,True,False
9,When are delivery addresses deleted from activ...,privacy_policy.txt,True,True,False


Expected-document hits: 12
Expected-fact hits: 12
Fallbacks: 2


### **Things to Note — policy RAG**
- **What is RAG?** Retrieval-Augmented Generation combines document retrieval with a language-model response grounded in the retrieved material.
- **What is chunking?** Splitting longer policy documents into smaller text sections so relevant passages can be retrieved.
- **What is TF-IDF?** A text representation that gives weight to terms based on their frequency in a document and how distinctive they are across documents.
- **What is cosine similarity?** A measure of similarity between vector directions; here it ranks policy chunks against the question.
- **Why retrieve policy before answering?** It supplies relevant source text so the assistant can base its answer on the policy rather than relying on unsupported assumptions.
- **What should happen when the policy does not cover a question?** The assistant should return its fallback response instead of inventing a rule.
- **What does the evaluation measure?** The displayed checks assess whether the expected document/fact was retrieved and whether the system used its fallback where appropriate.

## **8. Refund tools and safe stopping**
The refund agent demonstrates a controlled tool workflow:
1. Look up an order in SQLite.
2. Retrieve the relevant policy information.
3. Calculate the eligible refund.
4. Issue the simulated refund or escalate the case for human review.
The transaction ledger is a local JSONL file. This project is a simulation: no real payment or money movement takes place.

The tool map connects tool names to Python functions. Reviewing src/refund_agent.py helps trace how the agent selects and invokes those functions.

In [16]:
from src.refund_agent import get_order, search_policy, calculate_refund, TOOL_MAP, run_agent
print("Tools:", sorted(TOOL_MAP))
print("Example order lookup:")
with sqlite3.connect(db_path) as con:
    example_order = pd.read_sql_query(
        "SELECT order_id FROM orders WHERE status='completed' AND delivery_minutes > 45 LIMIT 1", con
    ).iloc[0, 0]
print(example_order, get_order(example_order))
print("Refund calculation:", calculate_refund(250.0, 52))
print("Timeout safety demo:", run_agent(f"Refund order {example_order}", scenario_id="timeout-demo", time_limit_seconds=0))

Tools: ['calculate_refund', 'escalate_to_human', 'get_order', 'issue_refund', 'search_policy']
Example order lookup:
ORD-100048 {'order_id': 'ORD-100048', 'status': 'completed', 'order_amount': 128.25, 'delivery_minutes': 65}
Refund calculation: {'refund_inr': 250.0}
Timeout safety demo: {'status': 'FALLBACK_TIMEOUT', 'final_answer': 'FALLBACK_TIMEOUT: hand off to a human agent.', 'steps': [], 'llm_calls': 0, 'tokens_used': 0}


In [17]:
from pathlib import Path
import pandas as pd

scenario_file = ROOT / "data" / "raw" / "agent_scenarios.csv"

scenarios = pd.read_csv(scenario_file)

print("Number of scenarios:", len(scenarios))
print("Available columns:", scenarios.columns.tolist())

display(scenarios)

Number of scenarios: 7
Available columns: ['scenario_id', 'customer_id', 'message', 'expected_outcome', 'expected_refund_inr']


,scenario_id,customer_id,message,expected_outcome,expected_refund_inr
0,S1,C0007,My order ORD-100048 reached me very late. I wo...,REFUND,128.25
1,S2,C0008,My order ORD-100067 reached me very late. I wo...,REFUND,135.00
2,S3,C0016,"Order ORD-100145 arrived far too late, please ...",ESCALATE,1999.00
3,S4,C0001,"Please refund order ORD-100001, I did not like...",DENIED,0.00
4,S5,C0003,I want my money back for order ORD-100025.,DENIED,0.00
5,S6,C0001,Refund for order ORD-999999 please.,NOT_FOUND,0.00
6,S7,C0007,[LOOP] Refund order ORD-100048.,FALLBACK,0.00


In [18]:
import json
from pathlib import Path
from src.refund_agent import run_agent

# Create output folders
outputs_dir = ROOT / "outputs"
outputs_dir.mkdir(parents=True, exist_ok=True)

state_dir = ROOT / "state"
state_dir.mkdir(parents=True, exist_ok=True)

agent_results = []

# Run all seven scenarios
for _, row in scenarios.iterrows():

    scenario_id = row["scenario_id"]
    user_request = row["message"]

    print(f"\nRunning {scenario_id}")
    print("Customer request:", user_request)

    result = run_agent(
        user_request,
        scenario_id=scenario_id,
        max_iterations=6,
        token_limit=6000,
        time_limit_seconds=10
    )

    record = {
        "scenario_id": scenario_id,
        "customer_id": row["customer_id"],
        "expected_outcome": row["expected_outcome"],
        "expected_refund_inr": row["expected_refund_inr"],
        "agent_result": result
    }

    agent_results.append(record)

# Save results as JSONL
agent_runs_file = outputs_dir / "agent_runs.jsonl"

with open(agent_runs_file, "w", encoding="utf-8") as f:
    for record in agent_results:
        f.write(json.dumps(record, default=str) + "\n")

print("\nAll seven scenarios processed.")
print("Agent results saved to:", agent_runs_file)


Running S1
Customer request: My order ORD-100048 reached me very late. I would like a refund.

Running S2
Customer request: My order ORD-100067 reached me very late. I would like a refund.

Running S3
Customer request: Order ORD-100145 arrived far too late, please refund me.

Running S4
Customer request: Please refund order ORD-100001, I did not like the delivery.

Running S5
Customer request: I want my money back for order ORD-100025.

Running S6
Customer request: Refund for order ORD-999999 please.

Running S7
Customer request: [LOOP] Refund order ORD-100048.

All seven scenarios processed.
Agent results saved to: outputs\agent_runs.jsonl


In [20]:
import json
from pathlib import Path

agent_runs_file = ROOT / "outputs" / "agent_runs.jsonl"

with open(agent_runs_file, "r", encoding="utf-8") as f:
    saved_results = [json.loads(line) for line in f if line.strip()]

print("Number of saved runs:", len(saved_results))

for record in saved_results:
    print("\nScenario:", record.get("scenario_id"))
    print("Expected:", record.get("expected_outcome"))

    result = record.get("agent_result", record)

    if isinstance(result, dict):
        print("Actual status:", result.get("status"))
        print("Final answer:", result.get("final_answer"))
        print("LLM calls:", result.get("llm_calls"))
        print("Tokens used:", result.get("tokens_used"))
    else:
        print("Agent result:", result)

Number of saved runs: 7

Scenario: S1
Expected: REFUND
Actual status: REFUND
Final answer: REFUND: INR 128.25 issued.
LLM calls: 5
Tokens used: 897

Scenario: S2
Expected: REFUND
Actual status: REFUND
Final answer: REFUND: INR 135.0 issued.
LLM calls: 5
Tokens used: 897

Scenario: S3
Expected: ESCALATE
Actual status: ESCALATE
Final answer: ESCALATE: refund above INR 1000 sent to a human lead.
LLM calls: 5
Tokens used: 884

Scenario: S4
Expected: DENIED
Actual status: DENIED
Final answer: DENIED: delivery was within 45 minutes.
LLM calls: 4
Tokens used: 605

Scenario: S5
Expected: DENIED
Actual status: DENIED
Final answer: DENIED: only completed orders can be refunded.
LLM calls: 3
Tokens used: 337

Scenario: S6
Expected: NOT_FOUND
Actual status: NOT_FOUND
Final answer: NOT_FOUND: no such order.
LLM calls: 2
Tokens used: 72

Scenario: S7
Expected: FALLBACK
Actual status: FALLBACK_MAX_ITERATIONS
Final answer: FALLBACK_MAX_ITERATIONS: hand off to a human agent.
LLM calls: 6
Tokens used: 3

In [21]:
for record in saved_results:
    result = record.get("agent_result", record)

    print(
        record["scenario_id"],
        "| Expected:", record["expected_outcome"],
        "| Actual:", result.get("status"),
        "| Answer:", result.get("final_answer")
    )

S1 | Expected: REFUND | Actual: REFUND | Answer: REFUND: INR 128.25 issued.
S2 | Expected: REFUND | Actual: REFUND | Answer: REFUND: INR 135.0 issued.
S3 | Expected: ESCALATE | Actual: ESCALATE | Answer: ESCALATE: refund above INR 1000 sent to a human lead.
S4 | Expected: DENIED | Actual: DENIED | Answer: DENIED: delivery was within 45 minutes.
S5 | Expected: DENIED | Actual: DENIED | Answer: DENIED: only completed orders can be refunded.
S6 | Expected: NOT_FOUND | Actual: NOT_FOUND | Answer: NOT_FOUND: no such order.
S7 | Expected: FALLBACK | Actual: FALLBACK_MAX_ITERATIONS | Answer: FALLBACK_MAX_ITERATIONS: hand off to a human agent.


In [22]:
from pathlib import Path

for folder_name in ["outputs", "state"]:
    folder = ROOT / folder_name

    print(f"\n--- {folder_name.upper()} ---")

    if folder.exists():
        files = list(folder.iterdir())

        if files:
            for file in files:
                print(file.name, "-", file.stat().st_size, "bytes")
        else:
            print("Folder is empty.")
    else:
        print("Folder does not exist.")


--- OUTPUTS ---
agent_runs.jsonl - 14882 bytes
customer_segments.csv - 43342 bytes
lapse_model.joblib - 600613 bytes
part_d_summary.json - 897 bytes
policy_eval_results.csv - 1274 bytes
policy_eval_zero_overlap.csv - 1276 bytes
refund_agent_eval.csv - 275 bytes
refund_ledger.jsonl - 869 bytes
retention_list.csv - 328 bytes
triage_budget_demo.csv - 11135 bytes
triage_results.csv - 11341 bytes

--- STATE ---
16a9c0a4-badc-451e-8f10-19b23d9f0b03.json - 3969 bytes
2c8fe85d-8f16-41b8-8609-03389f014d1a.json - 13601 bytes
3a6cfbc4-5f09-4280-b753-39d92f8ee1d0.json - 701 bytes
46f5b76c-f866-48bb-a026-bfed1ac0aecb.json - 701 bytes
66b6a263-650f-47b7-9b74-4d27e7b02ad0.json - 3535 bytes
673f19b8-16e2-4fc7-b950-5cbf1fffbfbb.json - 3957 bytes
795dd50b-fb1b-41a0-ab45-cad5b11fa2ad.json - 3094 bytes
9ddd2dff-9d40-4dc9-a599-a7a5b801f431.json - 3094 bytes
a0993e8f-b375-4374-972c-b067d529bae7.json - 13601 bytes
a4eaf96c-6859-4984-bd50-f6d1e25cd7ce.json - 3957 bytes
ab706844-d843-4b5f-8201-c2c455ea65ec.js

In [23]:
import json

ledger_file = ROOT / "outputs" / "refund_ledger.jsonl"

print("Refund ledger exists:", ledger_file.exists())

with open(ledger_file, "r", encoding="utf-8") as f:
    ledger_entries = [
        json.loads(line)
        for line in f
        if line.strip()
    ]

print("Number of ledger entries:", len(ledger_entries))

for entry in ledger_entries:
    print(entry)

Refund ledger exists: True
Number of ledger entries: 7
{'timestamp': '2026-11-07T13:24:40.103179+00:00', 'tool': 'issue_refund', 'order_id': 'ORD-100048', 'refund_inr': 128.25}
{'timestamp': '2026-11-07T13:24:40.117032+00:00', 'tool': 'issue_refund', 'order_id': 'ORD-100067', 'refund_inr': 135.0}
{'timestamp': '2026-11-07T13:24:40.123183+00:00', 'tool': 'escalate_to_human', 'order_id': 'ORD-100145', 'refund_inr': 1999.0}
{'timestamp': '2026-11-07T13:42:52.112668+00:00', 'tool': 'issue_refund', 'order_id': 'ORD-100048', 'refund_inr': 128.25}
{'timestamp': '2026-11-07T13:48:34.436051+00:00', 'tool': 'issue_refund', 'order_id': 'ORD-100048', 'refund_inr': 128.25}
{'timestamp': '2026-11-07T13:48:34.453501+00:00', 'tool': 'issue_refund', 'order_id': 'ORD-100067', 'refund_inr': 135.0}
{'timestamp': '2026-11-07T13:48:34.462880+00:00', 'tool': 'escalate_to_human', 'order_id': 'ORD-100145', 'refund_inr': 1999.0}


In [26]:
import importlib
import src.refund_agent as agent

importlib.reload(agent)

print("resume_agent available:", hasattr(agent, "resume_agent"))

resume_agent available: True


In [27]:
import uuid

test_run_id = str(uuid.uuid4())

try:
    agent.run_agent(
        "My order ORD-100048 reached me very late. I would like a refund.",
        scenario_id="S1-resume-test",
        run_id=test_run_id,
        crash_after=2
    )
except RuntimeError as e:
    print("Simulated crash:", e)

print(
    "Checkpoint saved:",
    (agent.STATE / f"{test_run_id}.json").exists()
)

Simulated crash: Simulated crash after checkpoint
Checkpoint saved: True


In [3]:
resumed_result = agent.resume_agent(test_run_id)

print("Final status:", resumed_result["status"])
print("Final answer:", resumed_result["final_answer"])
print("Total model calls:", resumed_result["llm_calls"])

print("\nTools executed:")
for step in resumed_result["steps"]:
    print(step["tool"])

Final status: REFUND
Final answer: REFUND: INR 128.25 issued.
Total model calls: 5

Tools executed:
get_order
search_policy
calculate_refund
issue_refund


### **Things to Note — refund tools and safe stopping**
- **What is a tool-calling agent?** An agent that chooses from a defined set of functions to retrieve information or perform a controlled action.
- **What is TOOL_MAP for?** It maps tool names to the Python functions that implement those tools.
- **Why look up the order first?** The agent needs the order's recorded details before applying a refund rule.
- **Why retrieve policy before calculating a refund?** Refund eligibility should be based on the applicable policy, not on an unsupported assumption.
-** What is the purpose of human escalation?** Cases that cannot be safely or confidently resolved automatically can be handed to a person.
- **Does this notebook issue real refunds?** No. The ledger is a local JSONL simulation; no money moves.
- **Why demonstrate a timeout?** It shows that the workflow should stop safely when it exceeds its allowed execution time rather than continuing indefinitely.

## **9. Generated artifacts and reproducibility**
The final step is to review the cleaned CSV files, database, model outputs, and state/ledger artifacts created by the pipeline.

To test reproducibility, remove the generated folders and database (data/clean/, data/instamart.db, and outputs/) and rerun the documented setup and notebook steps. The project should rebuild its outputs from the source generator and processing code.

Checkpoint note: if an assessment uses exact numeric checkpoints from a separate project brief, run those checks against the generated results and investigate any differences. Expected answers should not be hardcoded into the pipeline.

In [28]:
for folder in [ROOT/"data"/"clean", ROOT/"outputs", ROOT/"state"]:
    print(f"\\n{folder.relative_to(ROOT)}")
    print(sorted(p.name for p in folder.iterdir()) if folder.exists() else "(not created)")
print("Database exists:", db_path.exists())

\ndata\clean
['customers.csv', 'orders.csv', 'products.csv']
\noutputs
['agent_runs.jsonl', 'customer_segments.csv', 'lapse_model.joblib', 'part_d_summary.json', 'policy_eval_results.csv', 'policy_eval_zero_overlap.csv', 'refund_agent_eval.csv', 'refund_ledger.jsonl', 'retention_list.csv', 'triage_budget_demo.csv', 'triage_results.csv']
\nstate
['16a9c0a4-badc-451e-8f10-19b23d9f0b03.json', '2c8fe85d-8f16-41b8-8609-03389f014d1a.json', '3a6cfbc4-5f09-4280-b753-39d92f8ee1d0.json', '46f5b76c-f866-48bb-a026-bfed1ac0aecb.json', '66b6a263-650f-47b7-9b74-4d27e7b02ad0.json', '673f19b8-16e2-4fc7-b950-5cbf1fffbfbb.json', '795dd50b-fb1b-41a0-ab45-cad5b11fa2ad.json', '9ddd2dff-9d40-4dc9-a599-a7a5b801f431.json', 'a0993e8f-b375-4374-972c-b067d529bae7.json', 'a4eaf96c-6859-4984-bd50-f6d1e25cd7ce.json', 'ab706844-d843-4b5f-8201-c2c455ea65ec.json', 'ab840fac-45cc-4b41-8e0e-480530b82016.json', 'b6e81cac-35e1-4aed-ab45-4c3d56174651.json', 'e3f6632c-5e7a-4640-910c-c6f153aa41e8.json', 'f1a427b6-ff47-429f-b6

### **Things to Note — outputs and reproducibility**
- **What artifacts should be checked?** Clean CSV files, the SQLite database, model/output files, and state or ledger files created by the project.
- **What is reproducibility?** The ability to rebuild the same workflow outputs from the same source inputs and code.
- **Why delete generated outputs during the test?** It confirms that the notebook can recreate them instead of depending on stale files from an earlier run.
- **Why should expected values not be hardcoded?** Hardcoding hides errors and makes the pipeline less general; results should be calculated from the data and verified against the project requirements.
- **What is the overall project flow?** Generate source data → inspect and audit → clean and validate → store → analyse customers → predict lapse → triage tickets → retrieve policy → run controlled refund tools.

## **10. ReAct vs Plan-and-Execute**

### Comparison

**ReAct (Reason + Act)** combines reasoning and tool execution. The agent decides which tool to call based on the latest observation and continues until it reaches a final answer.

**Plan-and-Execute** first creates a complete plan and then executes the planned steps. It is more structured but may need replanning when unexpected results occur.

### Observations from our experiments

- S1 required 5 model calls to complete the refund.
- S5 required 3 model calls to deny an invalid refund.
- S6 required 2 model calls to identify a missing order.
- ReAct adjusts its next action based on tool observations.
- Plan-and-Execute can be more predictable for tasks with clearly defined steps, but unexpected tool results may require additional planning.

### Conclusion

For this customer support and refund use case, I prefer ReAct because the agent must verify order details, retrieve policies and make decisions based on tool responses. It provides flexibility when handling different customer situations. However, token limits, time limits and maximum iterations are important to prevent excessive execution.